# Giai đoạn 3 – Pipeline tiền xử lý thống nhất

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23010216-hieusphamj/text-summarization-nlp/blob/main/notebooks/03_preprocessing.ipynb)

**Mục tiêu**
1. Làm sạch CNN/DM (bỏ tiền tố rác `By . ... PUBLISHED:`, dateline `(CNN) --`...) và thống nhất định dạng với GOV.UK.
2. Lọc ngoại lệ **chỉ ở tập train** CNN/DM; kiểm tra rò rỉ giữa train và val/test.
3. **Đo bằng tokenizer thật** (BART, T5) tỉ lệ bài bị cắt ở 512 / 1024 token và tóm tắt bị cắt ở 64 / 128 token → chốt `max_input`, `max_target` bằng số liệu.
4. Dựng `Dataset` + `DataCollator` và in shape tensor (luồng dữ liệu cho báo cáo).

> Chạy từ trên xuống. Sau khi kernel khởi động lại, **luôn chạy lại cell 0 và cell 1** trước. Không cần GPU.

In [ ]:
# 0. Thiết lập môi trường
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/23010216-hieusphamj/text-summarization-nlp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("src").exists() and not Path("text-summarization-nlp").exists():
    subprocess.run(["git", "clone", REPO_URL], check=True)
if Path("text-summarization-nlp").exists():
    os.chdir("text-summarization-nlp")
while not Path("src").exists() and Path.cwd() != Path.cwd().parent:
    os.chdir("..")
sys.path.insert(0, os.getcwd())

DRIVE = None
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/nlp_summarization"); DRIVE.mkdir(parents=True, exist_ok=True)
print("Thư mục làm việc:", os.getcwd())

## 1. Chuẩn bị đầu vào (kết quả GĐ1 và GĐ2)
Các file `data/processed/*.jsonl` không nằm trên GitHub. Cell này khôi phục từ Drive nếu có, nếu không thì dựng lại (CNN/DM tải lại từ HuggingFace, GOV.UK dựng từ cache thô trên Drive) rồi lưu bản sao lên Drive cho các giai đoạn sau.

In [ ]:
import shutil, time
PROC = Path("data/processed"); PROC.mkdir(parents=True, exist_ok=True)
NEED = [f"{s}_{sp}.jsonl" for s in ("cnn_dm", "govuk") for sp in ("train", "validation", "test")]
missing = lambda: [f for f in NEED if not (PROC / f).exists()]

if missing() and DRIVE and (DRIVE / "processed").exists():
    for f in NEED:
        if (DRIVE / "processed" / f).exists():
            shutil.copy(DRIVE / "processed" / f, PROC / f)
    print("Đã khôi phục từ Drive; còn thiếu:", missing())

if any(f.startswith("cnn_dm") for f in missing()):
    from src.data.cnn_dm import main as build_cnn
    build_cnn("configs/data_cnn_dm.yaml")

if any(f.startswith("govuk") for f in missing()):
    raw = Path("data/raw/govuk"); raw.mkdir(parents=True, exist_ok=True)
    if not any(raw.glob("*.json")):
        drive_raw = DRIVE / "govuk_raw"
        assert drive_raw.exists(), "Không có cache thô GOV.UK trên Drive; hãy chạy notebook 02 trước."
        t0 = time.time()
        subprocess.run(f"cd {drive_raw} && ls | xargs -P 16 -I{{}} cp {{}} {raw.resolve()}/", shell=True, check=True)
        print(f"Đã chép {len(list(raw.glob('*.json')))} file cache ({time.time() - t0:.0f}s)")
    from src.crawl.build_dataset import main as build_gov
    build_gov("configs/data_govuk.yaml")

assert not missing(), missing()
if DRIVE:
    (DRIVE / "processed").mkdir(exist_ok=True)
    for f in NEED:
        shutil.copy(PROC / f, DRIVE / "processed" / f)
print("Sẵn sàng:", {f: sum(1 for _ in open(PROC / f, encoding="utf-8")) for f in NEED})

## 2. Làm sạch và lọc
Làm sạch áp dụng cho mọi tập. Lọc ngoại lệ (bài quá ngắn, tóm tắt quá ngắn/dài) **chỉ áp dụng cho train CNN/DM**, để val/test giữ nguyên phân bố thật khi so sánh với các công trình khác.

In [ ]:
import yaml, pandas as pd, numpy as np, matplotlib.pyplot as plt
from src.preprocessing.build_clean import main as run_clean, read_jsonl

CFG_PATH = "configs/preprocess.yaml"
cfg = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
FIG = Path(cfg["figures_dir"]); FIG.mkdir(parents=True, exist_ok=True)
TAB = Path(cfg["tables_dir"]);  TAB.mkdir(parents=True, exist_ok=True)

summary = run_clean(CFG_PATH)

### 2.1 Kiểm tra trước/sau khi làm sạch (CNN/DM, cả train, validation và test)

In [ ]:
import re
# Kiểm tra độc lập với các regex làm sạch: còn dấu vết ở 120 ký tự đầu không?
HEAD = {"By . (đầu bài)": r"^\s*By\s*\.", "PUBLISHED/UPDATED": r"PUBLISHED\s*:|UPDATED\s*:|Last updated at",
        "(CNN) ở đầu bài": r"\(CNN\)", "Scroll down for video": r"(?i)scroll down for"}
ANY = r"PUBLISHED\s*:|UPDATED\s*:|^\s*By\s*\.|\(CNN\)"
rows, left, changed_pct = [], {}, {}
for split in ["train", "validation", "test"]:
    raw_s = read_jsonl(Path(cfg["input_dir"]) / f"cnn_dm_{split}.jsonl")
    cln_s = {r["id"]: r for r in read_jsonl(Path(cfg["output_dir"]) / f"cnn_dm_{split}.jsonl")}
    pairs_s = [(r["article"], cln_s[r["id"]]["article"]) for r in raw_s if r["id"] in cln_s]
    if split == "train":
        pairs = pairs_s                      # dùng cho cell xem ví dụ bên dưới
    for name, pat in HEAD.items():
        rows.append({"split": split, "dấu vết": name,
                     "% trước": round(np.mean([bool(re.search(pat, a[:120])) for a, _ in pairs_s]) * 100, 2),
                     "% sau": round(np.mean([bool(re.search(pat, b[:120])) for _, b in pairs_s]) * 100, 2)})
    changed_pct[split] = np.mean([re.sub(r"\s+", " ", a).strip() != b for a, b in pairs_s]) * 100
    left[split] = [b[:160] for _, b in pairs_s if re.search(ANY, b[:120])]
chk = pd.DataFrame(rows); chk.to_csv(TAB / "cleaning_residual_check.csv", index=False)
print("Tỉ lệ bài bị thay đổi nội dung (ngoài khoảng trắng):", {k: f"{v:.1f}%" for k, v in changed_pct.items()})
print("Số bài còn sót dấu vết:", {k: len(v) for k, v in left.items()})
chk.pivot(index="dấu vết", columns="split", values=["% trước", "% sau"])

In [ ]:
# Xem ngẫu nhiên 6 mẫu (150 ký tự đầu) trước và sau, và các bài còn sót (nếu có)
rng = np.random.default_rng(1)
ex = []
for i in rng.choice(len(pairs), 6, replace=False):
    a, b = pairs[i]
    print("TRƯỚC:", repr(a[:150])); print("SAU  :", repr(b[:150])); print()
    ex.append({"truoc_150": a[:150], "sau_150": b[:150]})
pd.DataFrame(ex).to_csv(TAB / "cleaning_examples.csv", index=False)
for split, items in left.items():
    print(f"== {split}: còn sót {len(items)}")
    for x in items[:8]: print("  ", repr(x))

### 2.2 Lọc train, kiểm tra rò rỉ

In [ ]:
display(summary)
display(pd.read_csv(TAB / "leakage_check.csv"))

## 3. Đo độ dài token bằng tokenizer thật
Quyết định `max_input` / `max_target` phải dựa trên số liệu: bao nhiêu % bài bị cắt và bao nhiêu % token bị mất.
Với T5, đầu vào có thêm tiền tố `summarize: `.

In [ ]:
from transformers import AutoTokenizer
from src.preprocessing.tokenization import token_lengths, truncation_report

toks = {k: AutoTokenizer.from_pretrained(v) for k, v in cfg["tokenizers"].items()}
clean = {s: read_jsonl(Path(cfg["output_dir"]) / f"{s}_train.jsonl") for s in cfg["sources"]}
print({s: len(v) for s, v in clean.items()})

In [ ]:
rows, L = [], {}
for source, recs in clean.items():
    arts = [r["article"] for r in recs]
    sums = [r["highlights"].replace("\n", cfg["target_join"]) for r in recs]
    for tname, tok in toks.items():
        prefix = cfg["t5_prefix"] if tname == "t5" else ""
        L[(source, tname, "input")]  = token_lengths([prefix + a for a in arts], tok)
        L[(source, tname, "target")] = token_lengths(sums, tok)

for (source, tname, field), lens in L.items():
    thr = cfg["max_input_candidates"] if field == "input" else cfg["max_target_candidates"][source]
    pct = {p: int(np.percentile(lens, p)) for p in (50, 95, 99)}
    for r in truncation_report(lens, thr):
        rows.append({"source": source, "tokenizer": tname, "field": field, "p50": pct[50], "p95": pct[95], "p99": pct[99], **{k: round(v, 1) for k, v in r.items()}})
trunc = pd.DataFrame(rows)
trunc.to_csv(TAB / "token_truncation_report.csv", index=False)
trunc

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, source in zip(ax, ["cnn_dm", "govuk"]):
    lens = L[(source, "bart", "input")]
    a.hist(np.clip(lens, 0, 2500), bins=60, color="#4a6fa5")
    for t, c in zip(cfg["max_input_candidates"], ["r", "k"]):
        a.axvline(t, ls="--", c=c); a.text(t, a.get_ylim()[1] * 0.9, f" {t}", color=c)
    a.set_title(f"Độ dài đầu vào (token BART) – {source}"); a.set_xlabel("số token"); a.set_ylabel("số bài")
plt.tight_layout(); plt.savefig(FIG / "token_length_input.png", dpi=200); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for a, source in zip(ax, ["cnn_dm", "govuk"]):
    lens = L[(source, "bart", "target")]
    a.hist(np.clip(lens, 0, 250), bins=50, color="#3c8d53")
    for t in cfg["max_target_candidates"][source]:
        a.axvline(t, ls="--", c="r"); a.text(t, a.get_ylim()[1] * 0.9, f" {t}", color="r")
    a.set_title(f"Độ dài tóm tắt (token BART) – {source}"); a.set_xlabel("số token"); a.set_ylabel("số bài")
plt.tight_layout(); plt.savefig(FIG / "token_length_target.png", dpi=200); plt.show()

## 4. Luồng dữ liệu: Dataset → DataCollator → batch (shape tensor)

In [ ]:
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from src.preprocessing.tokenization import build_tokenized_dataset

MAX_IN, MAX_OUT = 512, 128
tok = toks["bart"]
ds = build_tokenized_dataset(clean["cnn_dm"][:200], tok, MAX_IN, MAX_OUT)
coll = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, return_tensors="pt")
batch = next(iter(DataLoader(ds, batch_size=4, collate_fn=coll)))
for k, v in batch.items():
    print(f"{k:<16} shape={tuple(v.shape)} dtype={v.dtype}")
print("\nĐầu vào (giải mã, 200 ký tự):", tok.decode(batch["input_ids"][0])[:200])
lab = batch["labels"][0]; lab = lab[lab != -100]
print("Nhãn (giải mã):", tok.decode(lab))

## 5. Nhận xét (điền sau khi chạy, rồi chép vào `docs/decisions.md` mục D7 và cập nhật D2)

- **Làm sạch:** ...% bài CNN/DM bị thay đổi nội dung; dấu vết ở đầu bài giảm từ ...% xuống ...% (bảng 2.1). Số bài còn sót: ...
- **Lọc train:** loại ... / 20.000 bài (...%). Val/test giữ nguyên.
- **Rò rỉ:** ... bài val/test trùng train (bảng 2.2).
- **max_input:** ...% bài CNN/DM bị cắt ở 512 token (mất ...% token), ...% ở 1024; GOV.UK: ...% ở 512. Chọn ... vì ...
- **max_target:** CNN/DM: ...% tóm tắt dài hơn 128 token; GOV.UK: ...% dài hơn 64 token. Chọn ...

In [ ]:
# 6. Lưu bản sao đã làm sạch lên Drive; (Colab) đóng gói kết quả để tải về
if DRIVE:
    (DRIVE / "clean").mkdir(exist_ok=True)
    for f in Path(cfg["output_dir"]).glob("*.jsonl"):
        shutil.copy(f, DRIVE / "clean" / f.name)
    print("Đã lưu dữ liệu sạch lên", DRIVE / "clean")
if IN_COLAB:
    from google.colab import files
    shutil.make_archive("phase3_outputs", "zip", ".", "results")
    files.download("phase3_outputs.zip")